# Language Models

A **language model** assigns probabilities to sequences of words. Given a partial sentence, it estimates which word comes next. Given a full sentence, it scores how likely that sequence is in the language. This single capability — modeling the probability distribution over text — underlies autocomplete, speech recognition, machine translation, and the large language models that power modern AI assistants.

Language modeling is one of the oldest tasks in NLP and one of the most impactful. From simple n-gram counts to billion-parameter Transformers, the goal has remained constant: learn $P(\text{word} \mid \text{context})$ and use it to understand, generate, and transform language.

This notebook covers the probabilistic foundations of language models, n-gram models built from scratch, evaluation with perplexity, neural and pre-trained language models, and text generation strategies.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. What Is a Language Model?

A language model (LM) is a function that assigns a probability to any sequence of words:

$$P(w_1, w_2, \ldots, w_n)$$

High probability means the sequence looks like natural language. Low probability means it is unlikely or ungrammatical.

**Applications that depend on language models:**

- **Autocomplete** — predict the next word as you type.
- **Speech recognition** — choose the most likely word sequence given audio.
- **Machine translation** — score candidate translations; prefer fluent output.
- **Spell checking** — *"teh"* is unlikely; *"the"* is probable.
- **Text generation** — produce new sentences by sampling from the model.
- **Downstream NLP** — pre-trained LMs provide representations for classification, QA, and summarization.

The central question: given words $w_1, \ldots, w_{t-1}$, what is the probability of the next word $w_t$?

$$P(w_t \mid w_1, w_2, \ldots, w_{t-1})$$

---

## 2. The Chain Rule of Probability

Any joint probability can be decomposed using the chain rule:

$$P(w_1, w_2, \ldots, w_n) = \prod_{t=1}^{n} P(w_t \mid w_1, \ldots, w_{t-1})$$

For the sentence *"the cat sat"*:

$$P(\text{the cat sat}) = P(\text{the}) \cdot P(\text{cat} \mid \text{the}) \cdot P(\text{sat} \mid \text{the cat})$$

Each factor is a **conditional probability** — the probability of the next word given all previous words. A language model learns to estimate each of these factors.

The challenge: the context $w_1, \ldots, w_{t-1}$ grows with every word. For long sentences, conditioning on the entire history is impractical — and no training corpus contains enough examples of every possible context.

In [ ]:
# Illustrating chain rule with estimated probabilities
probs = {
    "P(the)": 0.04,
    "P(cat | the)": 0.003,
    "P(sat | the cat)": 0.15,
}

joint = 1.0
print("Chain rule decomposition for 'the cat sat':\n")
for factor, p in probs.items():
    print(f"  {factor:<25s} = {p:.4f}")
    joint *= p

print(f"\n  P(the cat sat)          = {joint:.8f}")
print(f"  -log₂(P)                = {-np.log2(joint):.2f} bits")

---

## 3. N-gram Language Models

The **Markov assumption** simplifies the problem: each word depends only on the previous $n-1$ words, not the entire history.

**Unigram** ($n=1$): $P(w_t \mid w_{t-1}, \ldots) \approx P(w_t)$ — words are independent.

**Bigram** ($n=2$): $P(w_t \mid w_{t-1}, \ldots) \approx P(w_t \mid w_{t-1})$ — depends on one previous word.

**Trigram** ($n=3$): $P(w_t \mid \ldots) \approx P(w_t \mid w_{t-2}, w_{t-1})$ — depends on two previous words.

Estimation uses **maximum likelihood** from a training corpus — count how often each n-gram appears:

$$P(w_t \mid w_{t-n+1}, \ldots, w_{t-1}) = \frac{\text{count}(w_{t-n+1}, \ldots, w_t)}{\text{count}(w_{t-n+1}, \ldots, w_{t-1})}$$

Larger $n$ captures more context but requires exponentially more data. Most n-gram counts in a corpus are zero — creating the **sparsity problem**.

In [ ]:
corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat chased the dog",
    "a dog chased a cat",
    "the dog ran in the park",
    "a cat sat on a mat",
]

def tokenize(text):
    return ["<SOS>"] + re.findall(r"\b[a-z]+\b", text.lower()) + ["<EOS>"]

all_tokens = []
for doc in corpus:
    all_tokens.extend(tokenize(doc))

print(f"Tokenized corpus ({len(all_tokens)} tokens):")
print(f"  {all_tokens[:15]}...")

In [ ]:
class NGramLanguageModel:
    """N-gram language model with Laplace smoothing."""

    def __init__(self, n=2, alpha=1.0):
        self.n = n
        self.alpha = alpha  # smoothing parameter
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.vocab = set()

    def _get_ngrams(self, tokens):
        return [tuple(tokens[i:i+self.n]) for i in range(len(tokens) - self.n + 1)]

    def fit(self, documents):
        for doc in documents:
            tokens = tokenize(doc)
            self.vocab.update(tokens)
            for ngram in self._get_ngrams(tokens):
                self.ngram_counts[ngram] += 1
                self.context_counts[ngram[:-1]] += 1
        return self

    def probability(self, word, context):
        """P(word | context) with Laplace smoothing."""
        ngram = tuple(context) + (word,)
        count_ngram = self.ngram_counts.get(ngram, 0)
        count_context = self.context_counts.get(tuple(context), 0)
        vocab_size = len(self.vocab)
        return (count_ngram + self.alpha) / (count_context + self.alpha * vocab_size)

    def sentence_probability(self, text):
        tokens = tokenize(text)
        log_prob = 0.0
        for i in range(self.n - 1, len(tokens)):
            context = tokens[i - self.n + 1: i]
            word = tokens[i]
            p = self.probability(word, context)
            log_prob += np.log2(p + 1e-12)
        return 2 ** log_prob

    def next_word_probs(self, context):
        """Return P(w | context) for all vocabulary words."""
        return {w: self.probability(w, context) for w in self.vocab}


bigram_lm = NGramLanguageModel(n=2).fit(corpus)

print("Bigram probabilities P(w | context):")
contexts = [("the",), ("on",), ("chased",)]
for ctx in contexts:
    probs = bigram_lm.next_word_probs(ctx)
    top = sorted(probs.items(), key=lambda x: -x[1])[:5]
    ctx_str = " ".join(ctx)
    print(f"\n  P(w | '{ctx_str}'):")
    for w, p in top:
        print(f"    {w:8s} {p:.4f}")

---

## 4. Smoothing

Raw n-gram counts assign zero probability to unseen n-grams — any sentence containing a novel word combination gets probability zero. **Smoothing** redistributes probability mass to handle unseen events.

| Method | Idea |
|--------|------|
| **Laplace (add-one)** | Add 1 to every count: $P = \frac{\text{count} + 1}{N + V}$ |
| **Add-k smoothing** | Add $k < 1$ instead of 1 (less aggressive) |
| **Backoff** | If trigram unseen, fall back to bigram, then unigram |
| **Interpolation** | Weighted mix of unigram, bigram, trigram probabilities |
| **Kneser-Ney** | State-of-the-art for n-gram models; discounts counts and uses continuation probability |

Laplace smoothing is simple but overestimates the probability of unseen n-grams. Kneser-Ney smoothing is the standard for production n-gram language models.

In [ ]:
# Smoothing effect: unseen bigram "zebra" after "the"
for alpha in [0.0, 1.0, 0.1]:
    lm = NGramLanguageModel(n=2, alpha=alpha).fit(corpus)
    p_zebra = lm.probability("zebra", ("the",))
    p_cat = lm.probability("cat", ("the",))
    label = f"alpha={alpha}" if alpha > 0 else "no smoothing"
    print(f"{label:18s}  P(zebra|the)={p_zebra:.6f}  P(cat|the)={p_cat:.4f}")

---

## 5. Perplexity

**Perplexity** is the standard metric for evaluating language models. It measures how "surprised" the model is by a test sequence — lower is better.

$$\text{Perplexity} = P(w_1, \ldots, w_n)^{-\frac{1}{n}} = 2^{-\frac{1}{n} \sum_{t=1}^{n} \log_2 P(w_t \mid \text{context})}$$

Intuitively, perplexity is the **average branching factor** — how many equally likely choices the model considers at each step. A perplexity of 100 means the model is as uncertain as choosing uniformly among 100 words.

- Perplexity of 1 = perfect prediction (model always assigns probability 1 to the correct word).
- Higher perplexity = worse model.

Compare models on the **same test set** with the same vocabulary.

In [ ]:
def perplexity(lm, documents):
    total_log_prob = 0.0
    total_tokens = 0
    for doc in documents:
        tokens = tokenize(doc)
        for i in range(lm.n - 1, len(tokens)):
            context = tokens[i - lm.n + 1: i]
            word = tokens[i]
            p = lm.probability(word, context)
            total_log_prob += np.log2(p + 1e-12)
            total_tokens += 1
    return 2 ** (-total_log_prob / total_tokens)

train = corpus[:4]
test = corpus[4:]

print(f"{'Model':<15s} {'Perplexity':>12s}")
print("-" * 28)
for n in [1, 2, 3]:
    lm = NGramLanguageModel(n=n, alpha=1.0).fit(train)
    ppl = perplexity(lm, test)
    print(f"{n}-gram{' '*(10-len(str(n)))} {ppl:12.2f}")

---

## 6. Scoring Sentences

Language models score how probable a sentence is. Speech recognizers use this to choose between acoustic alternatives — *"recognize speech"* vs *"wreck a nice beach"* sound similar acoustically, but the former has much higher language model probability.

In [ ]:
lm = NGramLanguageModel(n=2, alpha=1.0).fit(corpus)

sentences = [
    "the cat sat on the mat",
    "the mat sat on the cat",
    "a dog chased a cat",
    "a cat chased a dog",
    "zebra elephant unicorn",
]

print(f"{'Sentence':<30s} {'Probability':>14s} {'Per-token log₂':>14s}")
print("-" * 60)
for sent in sentences:
    tokens = tokenize(sent)
    log_prob = 0.0
    for i in range(1, len(tokens)):
        p = lm.probability(tokens[i], (tokens[i-1],))
        log_prob += np.log2(p + 1e-12)
    prob = 2 ** log_prob
    avg_log = log_prob / (len(tokens) - 1)
    print(f"{sent:<30s} {prob:14.2e} {avg_log:14.2f}")

---

## 7. Text Generation with N-gram Models

To generate text, start with a seed context and repeatedly sample the next word from $P(w \mid \text{context})$. Append the sampled word to the context and continue until `<EOS>` or a maximum length.

In [ ]:
def generate(lm, seed="<SOS>", max_len=15):
    tokens = [seed] if seed != "<SOS>" else ["<SOS>"]
    for _ in range(max_len):
        context = tokens[-(lm.n - 1):]
        probs = lm.next_word_probs(tuple(context))
        words = list(probs.keys())
        p = np.array([probs[w] for w in words])
        p = p / p.sum()
        next_word = np.random.choice(words, p=p)
        if next_word == "<EOS>":
            break
        tokens.append(next_word)
    return " ".join(tokens[1:])  # skip <SOS>

lm = NGramLanguageModel(n=2, alpha=1.0).fit(corpus)

print("Generated sentences (bigram LM):")
for i in range(5):
    print(f"  {i+1}. {generate(lm)}")

---

## 8. Neural Language Models

N-gram models are limited by the Markov assumption and data sparsity. **Neural language models** use neural networks to learn distributed representations of context, generalizing to unseen word combinations.

Bengio et al. (2003) introduced the first neural LM: embed previous words, concatenate embeddings, pass through a feedforward network, output softmax over vocabulary.

RNN/LSTM language models (Mikolov et al., 2010) process context sequentially — the hidden state summarizes all previous words without a fixed window limit. This was the state of the art until Transformers.

$$P(w_t \mid w_1, \ldots, w_{t-1}) = \text{softmax}(\mathbf{W} \cdot \text{RNN}(w_1, \ldots, w_{t-1}))$$

Neural LMs share embeddings with downstream tasks — training a language model learns useful word representations as a byproduct (the foundation of Word2Vec and pre-training).

In [ ]:
# Character-level neural LM (simplified RNN)
text = "the cat sat on the mat the dog sat on the rug " * 3
chars = sorted(set(text))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
V = len(chars)

HIDDEN = 32
W_xh = np.random.randn(HIDDEN, V) * 0.01
W_hh = np.random.randn(HIDDEN, HIDDEN) * 0.01
W_hy = np.random.randn(V, HIDDEN) * 0.01

def rnn_forward_char(sequence, W_xh, W_hh, W_hy):
    h = np.zeros(HIDDEN)
    outputs = []
    for idx in sequence:
        x = np.zeros(V); x[idx] = 1.0
        h = np.tanh(W_xh @ x + W_hh @ h)
        logits = W_hy @ h
        exp_l = np.exp(logits - logits.max())
        outputs.append(exp_l / exp_l.sum())
    return outputs

seq = [char2idx[c] for c in text[:50]]
probs = rnn_forward_char(seq, W_xh, W_hh, W_hy)
print(f"Character-level LM: {V} characters, hidden dim {HIDDEN}")
print(f"Forward pass on {len(seq)} chars → {len(probs)} probability distributions")
print(f"Each distribution shape: {probs[0].shape} (vocab size)")

---

## 9. Pre-training Objectives

Modern language models are **pre-trained** on massive text corpora with self-supervised objectives, then fine-tuned for specific tasks.

**Causal language modeling (CLM)** — predict the next token given all previous tokens. Used by GPT, Llama, and other decoder-only models.

```
Input:  The cat sat on the
Target: cat sat on the mat
```

**Masked language modeling (MLM)** — predict randomly masked tokens using bidirectional context. Used by BERT.

```
Input:  The [MASK] sat on the mat
Target: cat
```

**Span corruption** — mask contiguous spans and train the model to reconstruct them. Used by T5.

Pre-training teaches grammar, facts, reasoning patterns, and world knowledge — all from raw text without manual labels.

---

## 10. Autoregressive Generation

Decoder-only models generate text **autoregressively** — one token at a time, each conditioned on all previously generated tokens.

$$P(w_1, \ldots, w_n) = \prod_{t=1}^{n} P(w_t \mid w_1, \ldots, w_{t-1})$$

```
Prompt: "The future of AI"
Step 1: P(w | "The future of AI") → sample "is"
Step 2: P(w | "The future of AI is") → sample "bright"
Step 3: P(w | "The future of AI is bright") → sample "."
Output: "The future of AI is bright."
```

The quality of generation depends heavily on the **sampling strategy** — how the next token is chosen from the probability distribution.

---

## 11. Sampling Strategies

Given a probability distribution over the vocabulary, several strategies select the next token:

**Greedy decoding** — always pick the highest-probability token. Deterministic but repetitive and prone to loops.

**Temperature sampling** — scale logits before softmax:

$$P(w_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- $T \to 0$: approaches greedy (deterministic).
- $T = 1$: sample from the model's distribution.
- $T > 1$: flatter distribution, more random and creative output.

**Top-k sampling** — sample only from the $k$ most likely tokens.

**Top-p (nucleus) sampling** — sample from the smallest set of tokens whose cumulative probability exceeds $p$ (e.g., 0.9).

**Beam search** — maintain $B$ highest-probability partial sequences at each step. Used in translation; less common in open-ended generation.

In [ ]:
def sample_with_temperature(logits, temperature=1.0):
  logits = np.array(logits, dtype=float)
  scaled = logits / max(temperature, 1e-8)
  exp_s = np.exp(scaled - scaled.max())
  return exp_s / exp_s.sum()

logits = np.array([2.0, 1.0, 0.5, 0.1, 0.05])  # simulated next-token logits
words = ["the", "a", "cat", "dog", "zebra"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, temp, title in zip(axes, [0.3, 1.0, 2.0], ["T=0.3 (focused)", "T=1.0 (default)", "T=2.0 (creative)"]):
    probs = sample_with_temperature(logits, temp)
    ax.bar(words, probs, color="steelblue")
    ax.set_title(title)
    ax.set_ylabel("Probability")
    ax.set_ylim(0, 1)
    for i, p in enumerate(probs):
        ax.text(i, p + 0.02, f"{p:.2f}", ha="center", fontsize=9)

plt.suptitle("Temperature controls randomness in generation", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def top_k_sample(logits, words, k=3):
    logits = np.array(logits)
    top_idx = np.argsort(logits)[-k:]
    top_logits = logits[top_idx]
    exp_l = np.exp(top_logits - top_logits.max())
    probs = exp_l / exp_l.sum()
    return [(words[i], probs[j]) for j, i in enumerate(top_idx)]

def top_p_sample(logits, words, p=0.9):
    logits = np.array(logits)
    exp_l = np.exp(logits - logits.max())
    probs = exp_l / exp_l.sum()
    sorted_idx = np.argsort(probs)[::-1]
    cumsum = 0.0
    selected = []
    for i in sorted_idx:
        cumsum += probs[i]
        selected.append(i)
        if cumsum >= p:
            break
    sel_probs = probs[selected]
    sel_probs = sel_probs / sel_probs.sum()
    return [(words[i], sel_probs[j]) for j, i in enumerate(selected)]

print("Top-k (k=3) candidates:")
for w, p in top_k_sample(logits, words, k=3):
    print(f"  {w:8s} {p:.3f}")

print("\nTop-p (p=0.9) candidates:")
for w, p in top_p_sample(logits, words, p=0.9):
    print(f"  {w:8s} {p:.3f}")

---

## 12. The Evolution of Language Models

| Era | Model | Key Idea | Scale |
|-----|-------|----------|-------|
| 1990s | N-gram | Count-based, Markov assumption | Millions of n-grams |
| 2003 | Neural LM (Bengio) | Distributed word representations | ~1M parameters |
| 2013 | Word2Vec | Efficient embedding learning | ~100M parameters |
| 2018 | BERT, GPT-1 | Transformer + pre-training | ~100M parameters |
| 2019 | GPT-2 | Scaling causal LM | 1.5B parameters |
| 2020 | GPT-3 | Few-shot learning emerges | 175B parameters |
| 2022+ | ChatGPT, Llama, Claude | Instruction tuning + RLHF | 7B–1T+ parameters |

Each generation reduced perplexity and expanded capabilities. Modern LLMs trained on trillions of tokens exhibit emergent abilities — reasoning, coding, translation — not explicitly programmed but arising from scale.

---

## 13. Large Language Models

**Large Language Models (LLMs)** are Transformer-based models trained on vast text corpora. They can:

- **Complete** text given a prompt.
- **Answer questions** across domains without task-specific training.
- **Follow instructions** after instruction fine-tuning (ChatGPT, Claude).
- **Write code**, analyze data, and engage in multi-turn dialogue.
- **Perform NLP tasks** via prompting — sentiment analysis, translation, summarization — without fine-tuning.

**Scaling laws** (Kaplan et al., 2020) show that model performance improves predictably with more parameters, data, and compute. This drove the race to ever-larger models.

**Challenges:** hallucination (generating false information), bias, high computational cost, difficulty guaranteeing factual accuracy, and environmental impact of training.

---

## 14. Using Language Models in Practice

**Prompting** — provide instructions and examples in natural language; the model performs the task without weight updates.

```
Classify the sentiment of this review as positive or negative:
Review: "The food was cold and the service was slow."
Sentiment:
```

**Fine-tuning** — update model weights on task-specific labeled data. More reliable than prompting for specialized domains.

**Retrieval-Augmented Generation (RAG)** — retrieve relevant documents from a knowledge base and include them in the prompt. Reduces hallucination for factual questions.

**Parameter-efficient fine-tuning (LoRA)** — train small adapter layers instead of the full model. Makes fine-tuning 7B+ models feasible on consumer hardware.

---

## 15. N-gram vs Neural vs LLM

| Aspect | N-gram LM | Neural LM (RNN/Transformer) | LLM (GPT-scale) |
|--------|-----------|----------------------------|----------------|
| **Context** | Fixed window ($n$ words) | Full sequence (limited by memory) | Very long context (8K–128K tokens) |
| **Generalization** | Poor (unseen n-grams get zero prob) | Good (embeddings capture similarity) | Excellent |
| **Data needed** | Moderate | Large | Massive (trillions of tokens) |
| **Perplexity** | High (100–1000+) | Medium (30–100) | Low (10–30) |
| **Generation quality** | Poor (local coherence only) | Moderate | High (coherent paragraphs) |
| **Compute** | Minimal | GPU required | Many GPUs / TPUs |

N-gram models remain useful for teaching, spell checking, and resource-constrained environments. Neural and LLM approaches dominate modern applications.

In [ ]:
# Perplexity comparison (illustrative ranges from literature)
models = ["Unigram", "Bigram", "Trigram", "RNN LM", "Transformer LM", "GPT-3"]
perplexities = [500, 200, 150, 80, 30, 20]  # illustrative on PTB-scale data

plt.figure(figsize=(9, 4))
bars = plt.bar(models, perplexities, color=["#aec7e8", "#98df8a", "#ffbb78",
                                              "#ff9896", "#c5b0d5", "#c49c94"])
plt.ylabel("Perplexity (lower is better)")
plt.title("Illustrative perplexity trends across LM generations")
plt.xticks(rotation=20, ha="right")
for bar, ppl in zip(bars, perplexities):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(ppl), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

---

## 16. Summary

A **language model** assigns probabilities to word sequences. The **chain rule** decomposes sentence probability into conditional next-word predictions. **N-gram models** estimate these with count-based statistics and smoothing; **perplexity** measures how well the model predicts held-out text.

**Neural language models** overcome n-gram sparsity with distributed representations. **Pre-trained Transformers** (BERT, GPT) trained on massive corpora with self-supervised objectives power modern NLP. **Autoregressive generation** produces text token by token; **temperature**, **top-k**, and **top-p** sampling control creativity and coherence.

Language modeling is the foundation of generative AI. Understanding probability, perplexity, and generation strategies is essential for building, evaluating, and deploying language models responsibly.